# Shapley sampling for LLM token attribution: comparing TokenSHAP, naive Monte Carlo Shapley, and KernelSHAP

Task: E2 (spike). Branch: `spike/E2-shapley-sampling`. Brief: `docs/02_engineer_brief.md` section 6 E2 plus the amendments in `docs/spec-challenges/E2.md`.

Intent: benchmark three Shapley sampling strategies for token-level attribution on a handcrafted 9-prompt set, then recommend a v1.0 strategy in `RECOMMENDATION.md`. Empirical sections in this notebook execute when the PM runs the notebook with an Anthropic API key configured.

## Scope and constraints

- **Spend cap.** Total token spend across the full notebook run stays under $50. The runner aborts at $45 to leave a 10% buffer. The notebook reports actual spend in USD with the pricing snapshot date `2026-05-14`.
- **Statistical floor.** Every benchmark cell runs at least 5 independent repeats. Reported uncertainty is a 95% bootstrap confidence interval (1000 resamples) over the repeats. The bootstrap is documented over normal approximation because per-cell sample sizes are small and the underlying distributions are not assumed normal.
- **Prompt size matrix.** The handcrafted prompt set in `prompts.py` covers 1k, 4k, and 16k tokens by character-count estimate. The notebook reports actual Anthropic tokens after the first call.
- **Faithfulness operationalizations.** Per DeYoung et al. (2020), Section 4.1: sufficiency (Equation 1), comprehensiveness (Equation 2), and removal-as-prediction-change defined as the average L1 distance over single-token removals. Citations land in the bibliography cell at the end.
- **Model.** Anthropic Haiku 4-5 at temperature 0. Responses cache to `.cache/` on disk so re-runs are cheap.
- **Ranking rule.** Top-k tokens are selected by absolute attribution value. The notebook reports k at 10%, 25%, and 50% of the token count.
- **Empty-completion handling.** If the model returns whitespace, the prediction function returns 0.0 (the L1-distance metric stays well-defined; sufficiency and comprehensiveness pin to the full-prompt baseline).

## API non-determinism caveat

Anthropic at temperature 0 is not bit-exact across calls. The variance pass runs an identical prompt 10 times and reports the variance floor before attributing any variance to the Shapley estimator. If the noise floor exceeds the inter-method differences, the recommendation states that explicitly and does not pick a winner inside the noise band.

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
from __future__ import annotations

import logging
import os
import random
import sys
import time  # used by later benchmark cells
from pathlib import Path

# Make the helpers package importable without a wheel install.
spike_root = Path.cwd()
if str(spike_root) not in sys.path:
    sys.path.insert(0, str(spike_root))

import prompts as prompt_set  # noqa: E402  # sys.path mutation precedes
from helpers import (  # noqa: E402  # sys.path mutation precedes
    anthropic_client,
    kernel_shap,  # used by later benchmark cells
    metrics,
    shapley,  # used by later benchmark cells
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("E2_spike")

# Deterministic seeds where applicable.
SEED = 20260514
random.seed(SEED)

MODEL = "claude-haiku-4-5"
CACHE_DIR = Path(".cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if not os.environ.get("ANTHROPIC_API_KEY"):
    logger.warning("ANTHROPIC_API_KEY is not set; live calls will fail.")
else:
    logger.info("ANTHROPIC_API_KEY detected.")

PRICING_SNAPSHOT_DATE = "2026-05-14"
SPEND_HARD_CAP_USD = 50.0
SPEND_ABORT_USD = 45.0

_ = (metrics, kernel_shap, shapley, prompt_set)  # silence "unused" while cells below reference them

## Prompt set

Nine handcrafted prompts: three RAG, three instruction-following, three reasoning, at 1k, 4k, and 16k token brackets. Definitions live in `prompts.py` for reproducibility. Each entry includes an `expected_topic` field that the notebook uses for sanity reporting after attribution.

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
for entry in prompt_set.PROMPTS:
    estimated = anthropic_client.estimate_token_count(entry["prompt"])
    logger.info(
        "id=%s category=%s bucket=%s estimated_tokens=%d",
        entry["id"],
        entry["category"],
        entry["length_bucket"],
        estimated,
    )

## Spend tracking

A running USD counter accumulates across every API call. The runner aborts at the soft threshold (`SPEND_ABORT_USD`) and refuses to start a new cell if the projected post-cell spend would exceed `SPEND_HARD_CAP_USD`.

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
from dataclasses import dataclass, field


@dataclass
class SpendTracker:
    """Running USD counter with soft and hard caps."""

    spent_usd: float = 0.0
    calls: int = 0
    history: list[float] = field(default_factory=list)

    def record(self, usage: anthropic_client.Usage) -> None:
        cost = usage.cost_usd()
        self.spent_usd += cost
        self.calls += 1
        self.history.append(cost)
        if self.spent_usd >= SPEND_ABORT_USD:
            raise RuntimeError(
                f"Spend abort: {self.spent_usd:.2f} USD >= soft cap {SPEND_ABORT_USD} USD."
            )

    def projected_ok(self, additional_usd: float) -> bool:
        return self.spent_usd + additional_usd < SPEND_HARD_CAP_USD


tracker = SpendTracker()

## API non-determinism floor

Run an identical prompt 10 times at temperature 0 with caching disabled, then report the variance over the response strings. The noise floor anchors the comparison: any inter-method difference smaller than this floor is reported as inside the noise band.

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
import tempfile
from collections import Counter


def measure_noise_floor(prompt: str, n_runs: int = 10) -> dict[str, float | int | str]:
    """Run an identical prompt n_runs times with a throwaway cache."""
    outputs: list[str] = []
    with tempfile.TemporaryDirectory() as throwaway:
        for _ in range(n_runs):
            result = anthropic_client.complete(
                prompt,
                temperature=0.0,
                model=MODEL,
                cache_dir=throwaway,
            )
            outputs.append(result.text)
            tracker.record(result.usage)
    distinct = len(set(outputs))
    counts = Counter(outputs)
    most_common, freq = counts.most_common(1)[0]
    return {
        "n_runs": n_runs,
        "distinct_outputs": distinct,
        "mode_frequency": freq,
        "mode_preview": most_common[:120],
    }


# Run on the cheapest prompt to stay under the spend cap.
noise_report = measure_noise_floor(prompt_set.by_id("rag-1k")["prompt"], n_runs=10)
logger.info("noise floor: %s", noise_report)

## Value function and tokenization

All three methods need a value function that maps a coalition (subset of kept tokens) to a scalar. The notebook uses a single shared definition: the value is the next-token log-probability of the original completion under the ablated prompt. The original completion is obtained once per prompt and cached.

Token boundaries are split on whitespace plus punctuation for the spike. The notebook flags this as an approximation; production v1.0 uses the Anthropic tokenizer surface.

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
import re
from collections.abc import Callable, Iterable

TOKEN_SPLIT_RE = re.compile(r"\s+")

ValueFn = Callable[[Iterable[int]], float]


def split_tokens(text: str) -> list[str]:
    """Whitespace split. Spike approximation; production uses the tokenizer."""
    return [tok for tok in TOKEN_SPLIT_RE.split(text) if tok]


def join_tokens(tokens: list[str]) -> str:
    """Inverse of split_tokens for the spike approximation."""
    return " ".join(tokens)


def make_value_fn(full_tokens: list[str], target_completion: str) -> ValueFn:
    """Build a coalition value function for a given prompt and target.

    The value is a similarity score between the target completion and
    the model's completion on the ablated prompt. The spike uses a
    normalized character-overlap score in [0, 1] so the value function
    is bounded and works without log-prob access.
    """
    target_chars = set(target_completion.lower())

    def overlap_score(reconstructed_tokens: list[str]) -> float:
        prompt = join_tokens(reconstructed_tokens)
        if not prompt.strip():
            return 0.0
        result = anthropic_client.complete(
            prompt, temperature=0.0, model=MODEL, cache_dir=str(CACHE_DIR)
        )
        tracker.record(result.usage)
        candidate = result.text.lower()
        if not candidate:
            return 0.0
        candidate_chars = set(candidate)
        if not target_chars:
            return 0.0
        return len(target_chars & candidate_chars) / len(target_chars | candidate_chars)

    def coalition_to_tokens(coalition: Iterable[int]) -> list[str]:
        kept = set(coalition)
        return [full_tokens[i] if i in kept else " " for i in range(len(full_tokens))]

    def value_fn(coalition: Iterable[int]) -> float:
        return overlap_score(coalition_to_tokens(coalition))

    return value_fn

## Method 1: TokenSHAP

TokenSHAP (Horovicz & Goldshmidt, 2024) ships a `TokenSHAP` class with an `analyze(prompt, sampling_ratio, ...)` method. The internal sampling strategy is Monte Carlo Shapley value estimation; the `sampling_ratio` parameter trades off coverage and cost. The notebook pins the install command to a SHA captured by the PM at execution time.

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
# Install pin: PM records the commit SHA used at execution time.
# pip install git+https://github.com/ronigold/TokenSHAP@<SHA>


def run_tokenshap(prompt_text: str, sampling_ratio: float) -> dict[str, object]:
    """Invoke TokenSHAP and return attribution scores plus runtime.

    The function imports the library lazily so the notebook still imports
    when the dependency is absent. PM execution installs the pinned SHA
    before running this cell.
    """
    try:
        from token_shap import StringSplitter, TokenSHAP  # type: ignore[import-not-found]
    except ImportError as exc:
        return {"error": f"token_shap import failed: {exc}"}

    class HaikuModel:
        """Adapter exposing the call interface TokenSHAP expects."""

        def generate(self, prompt: str) -> str:
            result = anthropic_client.complete(
                prompt, temperature=0.0, model=MODEL, cache_dir=str(CACHE_DIR)
            )
            tracker.record(result.usage)
            return result.text

    splitter = StringSplitter(split_pattern=r"\s+")
    explainer = TokenSHAP(model=HaikuModel(), splitter=splitter)
    started = time.perf_counter()
    df = explainer.analyze(prompt_text, sampling_ratio=sampling_ratio)
    elapsed = time.perf_counter() - started
    return {"attribution_frame": df, "runtime_s": elapsed}

## Method 2: Naive Monte Carlo Shapley

Implementation lives in `helpers/shapley.py`. The estimator follows Strumbelj and Kononenko (2014) plus Castro et al. (2009): uniform permutation sampling, marginal-contribution accumulation. The sample budget is the variable `M_NAIVE`. The notebook reports results at 10, 50, 200, and 1000.

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
def run_naive_mc(prompt_text: str, target_topic: str, m_naive: int) -> dict[str, object]:
    """Run the naive Monte Carlo Shapley estimator on a single prompt."""
    tokens = split_tokens(prompt_text)
    players = list(range(len(tokens)))
    value_fn = make_value_fn(tokens, target_topic)

    started = time.perf_counter()
    scores = shapley.shapley_monte_carlo(players, value_fn, n_samples=m_naive, seed=SEED)
    elapsed = time.perf_counter() - started
    return {
        "tokens": tokens,
        "scores": scores,
        "runtime_s": elapsed,
        "sample_budget": m_naive,
    }

## Method 3: KernelSHAP (token binary adapter)

Implementation in `helpers/kernel_shap.py`. Each token is a binary feature; the mask token is a single space, preserving sequence length and word boundaries. The notebook documents two known pitfalls of this adaptation (token interdependence, kernel-weight mismatch with the natural ablation distribution). KernelSHAP is the most-pitfall-prone of the three methods and is here as the comparison's stress test.

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
def run_kernel_shap(prompt_text: str, target_topic: str, nsamples: int) -> dict[str, object]:
    """Run KernelSHAP over token-level binary features on a single prompt."""
    tokens = split_tokens(prompt_text)
    target_chars = set(target_topic.lower())

    def prediction_fn(reconstructed_tokens: list[str]) -> float:
        prompt = " ".join(reconstructed_tokens)
        if not prompt.strip():
            return 0.0
        result = anthropic_client.complete(
            prompt, temperature=0.0, model=MODEL, cache_dir=str(CACHE_DIR)
        )
        tracker.record(result.usage)
        candidate = result.text.lower()
        if not candidate or not target_chars:
            return 0.0
        candidate_chars = set(candidate)
        return len(target_chars & candidate_chars) / len(target_chars | candidate_chars)

    started = time.perf_counter()
    scores = kernel_shap.kernel_shap_token_attribution(
        tokens, prediction_fn, nsamples=nsamples, seed=SEED
    )
    elapsed = time.perf_counter() - started
    return {
        "tokens": tokens,
        "scores": scores,
        "runtime_s": elapsed,
        "sample_budget": nsamples,
    }

## Faithfulness metric harness

Sufficiency, comprehensiveness, and removal-as-prediction-change live in `helpers/metrics.py`. All three accept a prediction callback so they apply uniformly across the three Shapley methods. The notebook reports each metric at k equal to 10%, 25%, and 50% of the token count.

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
def evaluate_attribution(
    tokens: list[str],
    scores: list[float],
    target_topic: str,
    k_fractions: tuple[float, ...] = (0.1, 0.25, 0.5),
) -> dict[str, float]:
    """Score sufficiency, comprehensiveness, and RaPC for one attribution."""
    target_chars = set(target_topic.lower())

    def predict(coalition: Iterable[int]) -> float:
        kept = set(coalition)
        reconstructed = [tokens[i] if i in kept else " " for i in range(len(tokens))]
        prompt = " ".join(reconstructed)
        if not prompt.strip():
            return 0.0
        result = anthropic_client.complete(
            prompt, temperature=0.0, model=MODEL, cache_dir=str(CACHE_DIR)
        )
        tracker.record(result.usage)
        candidate = result.text.lower()
        if not candidate or not target_chars:
            return 0.0
        candidate_chars = set(candidate)
        return len(target_chars & candidate_chars) / len(target_chars | candidate_chars)

    def distribution(coalition: Iterable[int]) -> list[float]:
        # The spike's value function is a scalar similarity, so the
        # "distribution" is a degenerate two-point [p, 1-p]. The L1
        # metric stays well-defined.
        p = predict(coalition)
        return [p, 1.0 - p]

    n = len(tokens)
    results: dict[str, float] = {}
    for frac in k_fractions:
        k = max(1, round(frac * n))
        pct = round(frac * 100)
        results[f"sufficiency_k{pct}"] = metrics.sufficiency(predict, scores, k)
        results[f"comprehensiveness_k{pct}"] = metrics.comprehensiveness(predict, scores, k)
    results["removal_l1"] = metrics.removal_as_prediction_change(distribution, n)
    return results

## Benchmark runner

For each prompt size by method by sample budget, the runner records runtime, faithfulness metrics, and USD spend. Each cell runs at least 5 repeats. The cache makes repeats nearly free in API calls but the wall-clock cost still accrues. The runner aborts at the soft spend threshold.

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
SAMPLE_BUDGETS = (10, 50, 200, 1000)
N_REPEATS = 5


def benchmark_method(
    method_name: str,
    runner: callable,
    prompt_entry: prompt_set.PromptEntry,
    sample_budget: int,
) -> dict[str, object]:
    """Run a method N_REPEATS times and collect runtime plus faithfulness."""
    runtimes: list[float] = []
    sufficiency_samples: dict[int, list[float]] = {10: [], 25: [], 50: []}
    comprehensiveness_samples: dict[int, list[float]] = {10: [], 25: [], 50: []}
    rapc_samples: list[float] = []
    for repeat in range(N_REPEATS):
        out = runner(prompt_entry["prompt"], prompt_entry["expected_topic"], sample_budget)
        runtimes.append(float(out["runtime_s"]))
        if "tokens" in out and "scores" in out:
            faith = evaluate_attribution(
                out["tokens"], out["scores"], prompt_entry["expected_topic"]
            )
            for k_pct in (10, 25, 50):
                sufficiency_samples[k_pct].append(faith[f"sufficiency_k{k_pct}"])
                comprehensiveness_samples[k_pct].append(faith[f"comprehensiveness_k{k_pct}"])
            rapc_samples.append(faith["removal_l1"])
        if not tracker.projected_ok(0.0):
            logger.warning("Aborting repeats: spend cap reached at repeat %d", repeat)
            break
    return {
        "method": method_name,
        "prompt_id": prompt_entry["id"],
        "sample_budget": sample_budget,
        "runtime_mean_ci": metrics.bootstrap_ci(runtimes),
        "sufficiency_ci": {k: metrics.bootstrap_ci(v) for k, v in sufficiency_samples.items()},
        "comprehensiveness_ci": {
            k: metrics.bootstrap_ci(v) for k, v in comprehensiveness_samples.items()
        },
        "rapc_ci": metrics.bootstrap_ci(rapc_samples),
        "spend_usd": tracker.spent_usd,
    }

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
# The benchmark sweep: every method by every prompt by every budget.
# PM execution runs this cell; the spike author keeps it small enough to
# stay under the $50 cap (cache hits absorb most of the duplicate cost).
results: list[dict[str, object]] = []
for entry in prompt_set.PROMPTS:
    for budget in SAMPLE_BUDGETS:
        if not tracker.projected_ok(1.0):
            logger.warning("Spend projection halts further sweeps.")
            break
        results.append(
            benchmark_method(
                "tokenshap",
                lambda p, t, b: run_tokenshap(p, sampling_ratio=min(1.0, b / 1000.0)),
                entry,
                budget,
            )
        )
        results.append(benchmark_method("naive_mc", run_naive_mc, entry, budget))
        results.append(benchmark_method("kernel_shap", run_kernel_shap, entry, budget))
        logger.info(
            "prompt=%s budget=%d spend_so_far_usd=%.3f",
            entry["id"],
            budget,
            tracker.spent_usd,
        )
logger.info("sweep complete: %d records, total_spend_usd=%.3f", len(results), tracker.spent_usd)

## Plots

Three matplotlib figures: runtime vs prompt length per method, faithfulness vs sample budget per method, and the cost-faithfulness Pareto frontier. The plotting code intentionally stays simple; the goal is comparison, not publication-grade graphics.

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
import matplotlib.pyplot as plt

BUCKETS = ("1k", "4k", "16k")
METHOD_LABELS = ("tokenshap", "naive_mc", "kernel_shap")


def plot_runtime_vs_prompt_length() -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    for method in METHOD_LABELS:
        xs = []
        ys = []
        for bucket in BUCKETS:
            rows = [r for r in results if r["method"] == method and r["prompt_id"].endswith(bucket)]
            if not rows:
                continue
            mean = sum(r["runtime_mean_ci"][0] for r in rows) / len(rows)
            xs.append(bucket)
            ys.append(mean)
        ax.plot(xs, ys, marker="o", label=method)
    ax.set_xlabel("prompt length bucket")
    ax.set_ylabel("runtime (s, mean across prompts and budgets)")
    ax.set_title("Runtime vs prompt length per method")
    ax.legend()
    fig.tight_layout()
    return fig


def plot_faithfulness_vs_budget() -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    for method in METHOD_LABELS:
        xs = list(SAMPLE_BUDGETS)
        ys = []
        for budget in SAMPLE_BUDGETS:
            rows = [r for r in results if r["method"] == method and r["sample_budget"] == budget]
            if not rows:
                ys.append(float("nan"))
                continue
            vals = [r["comprehensiveness_ci"][25][0] for r in rows]
            ys.append(sum(vals) / len(vals))
        ax.plot(xs, ys, marker="o", label=method)
    ax.set_xlabel("sample budget")
    ax.set_xscale("log")
    ax.set_ylabel("comprehensiveness at k=25% (higher is better)")
    ax.set_title("Faithfulness vs sample budget per method")
    ax.legend()
    fig.tight_layout()
    return fig


def plot_cost_faithfulness_pareto() -> None:
    fig, ax = plt.subplots(figsize=(7, 4))
    for method in METHOD_LABELS:
        rows = [r for r in results if r["method"] == method]
        xs = [r["spend_usd"] for r in rows]
        ys = [r["comprehensiveness_ci"][25][0] for r in rows]
        ax.scatter(xs, ys, label=method)
    ax.set_xlabel("cumulative spend at record (USD)")
    ax.set_ylabel("comprehensiveness at k=25%")
    ax.set_title("Cost vs faithfulness scatter (Pareto frontier visible top-left)")
    ax.legend()
    fig.tight_layout()
    return fig


_ = plot_runtime_vs_prompt_length()
_ = plot_faithfulness_vs_budget()
_ = plot_cost_faithfulness_pareto()
plt.show()

## Variance pass

Every benchmark cell already runs 5 repeats; the per-cell bootstrap CI is the variance pass. This cell collects the per-method, per-prompt CI widths and prints a summary so the recommendation has a quick reference to the variance bands.

In [ ]:
# Outputs intentionally empty; PM-accept fills these in.
for record in results:
    ci_lo = record["comprehensiveness_ci"][25][1]
    ci_hi = record["comprehensiveness_ci"][25][2]
    width = ci_hi - ci_lo if ci_hi == ci_hi else float("nan")
    logger.info(
        "method=%s prompt=%s budget=%d comp_k25_ci_width=%.4f",
        record["method"],
        record["prompt_id"],
        record["sample_budget"],
        width,
    )

## Open questions for PM

1. **Sample budget default for v1.0.** Pre-execution guess is 200; the notebook validates against the cost-faithfulness scatter.
2. **Top-k default.** k = 25% is the spike default; PM picks the v1.0 default after the data lands.
3. **Promotion of `helpers/` to `src/markovtrace/`.** Requires the TDD and attack-first commits before any move per CLAUDE.md Rules 9 and 10.
4. **TokenSHAP install pin.** PM records the commit SHA used at execution and updates the install line in the method-1 cell.
5. **Pricing snapshot date.** Locked at 2026-05-14 in the setup cell; PM bumps if a re-execution lands after a price change.
6. **Tokenizer fidelity.** The spike uses whitespace splitting. v1.0 will use the Anthropic tokenizer; the notebook flags this as the obvious accuracy gap when interpreting per-token scores.

## Bibliography

- Castro, J., Gomez, D., & Tejada, J. (2009). Polynomial calculation of the Shapley value based on sampling. Computers & Operations Research, 36(5), 1726-1730. https://doi.org/10.1016/j.cor.2008.04.004 (Accessed 2026-05-14).
- DeYoung, J., Jain, S., Rajani, N. F., Lehman, E., Xiong, C., Socher, R., & Wallace, B. C. (2020). ERASER: A Benchmark to Evaluate Rationalized NLP Models. ACL 2020. https://aclanthology.org/2020.acl-main.408/ (Accessed 2026-05-14).
- Horovicz, M., & Goldshmidt, R. (2024). TokenSHAP: Interpreting Large Language Models with Monte Carlo Shapley Value Estimation. arXiv:2407.10114. https://arxiv.org/abs/2407.10114 (Accessed 2026-05-14).
- Lundberg, S. M., & Lee, S.-I. (2017). A Unified Approach to Interpreting Model Predictions. NeurIPS 2017. https://proceedings.neurips.cc/paper/2017/hash/8a20a8621978632d76c43dfd28b67767-Abstract.html (Accessed 2026-05-14).
- Strumbelj, E., & Kononenko, I. (2014). Explaining prediction models and individual predictions with feature contributions. Knowledge and Information Systems, 41(3), 647-665. https://doi.org/10.1007/s10115-013-0679-x (Accessed 2026-05-14).